#### Importing libraries and loading the dataset 

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [6]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    r"C:\Users\user\OneDrive - United States International University (USIU)\Desktop\FS2026\DSA4060-Austin Odera(Recommender Systems)\Book recomender system"
)

USERS_PATH = BASE_DIR / "BX-Users.csv"
BOOKS_PATH = BASE_DIR / "BX-Books.csv"
RATINGS_PATH = BASE_DIR / "BX-Book-Ratings.csv"


def load_bx_csv(path: Path) -> pd.DataFrame:
    # Try common delimiters, encodings, and robust bad-line handling
    for sep in [";", ",", "\t"]:
        for enc in ["latin1", "utf-8", "cp1252"]:
            try:
                df = pd.read_csv(
                    path,
                    sep=sep,
                    encoding=enc,
                    low_memory=False,
                    on_bad_lines="skip",  # skips corrupted/multiline quotes
                    escapechar="\\",
                )
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue

    # Fallback with python engine if C-parser fails
    return pd.read_csv(
        path,
        sep=None,
        engine="python",
        encoding="latin1",
        on_bad_lines="skip",
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    for col in df.columns:
        key = str(col).strip().lower().replace("-", "_").replace(" ", "_").replace('"', "")
        if key in {"user_id", "userid"}:
            rename_map[col] = "user_id"
        elif key in {"isbn", "book_id", "item_id", "itemid"}:
            rename_map[col] = "item_id"
        elif key in {"book_rating", "rating"}:
            rename_map[col] = "rating"
        elif key in {"book_title", "title"}:
            rename_map[col] = "title"
        elif key in {"book_author", "author"}:
            rename_map[col] = "author"
        elif key in {"year_of_publication", "year"}:
            rename_map[col] = "year"
        elif key == "publisher":
            rename_map[col] = "publisher"
        elif key == "location":
            rename_map[col] = "location"
        elif key == "age":
            rename_map[col] = "age"
    return df.rename(columns=rename_map)


users_raw = normalize_columns(load_bx_csv(USERS_PATH))
books_raw = normalize_columns(load_bx_csv(BOOKS_PATH))
ratings_raw = normalize_columns(load_bx_csv(RATINGS_PATH))

print("Raw shapes -> Users:", users_raw.shape, "| Books:", books_raw.shape, "| Ratings:", ratings_raw.shape)
print("Normalized columns -> Users:", list(users_raw.columns))
print("Normalized columns -> Books:", list(books_raw.columns))
print("Normalized columns -> Ratings:", list(ratings_raw.columns))

Raw shapes -> Users: (278858, 3) | Books: (271379, 8) | Ratings: (1149780, 3)
Normalized columns -> Users: ['user_id', 'location', 'age']
Normalized columns -> Books: ['item_id', 'title', 'author', 'year', 'publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']
Normalized columns -> Ratings: ['user_id', 'item_id', 'rating']


Book descriptions / plot synopses / genres are item metadata (content features), not interaction signals. They describe what the book is about so you can compute item similarity or enrich features, whereas interactions (user_id, item_id, rating/event) capture who interacted with what.

#### Cleaning and sparsity 

Clean ratings: keep explicit ratings (1-10) and drop missing/duplicate pairs

In [7]:
ratings_df = ratings_raw[["user_id", "item_id", "rating"]].dropna().copy()
ratings_df["user_id"] = ratings_df["user_id"].astype(str).str.strip()
ratings_df["item_id"] = ratings_df["item_id"].astype(str).str.strip()
ratings_df["rating"] = pd.to_numeric(ratings_df["rating"], errors="coerce")
ratings_df = ratings_df.dropna(subset=["rating"])
ratings_df = ratings_df[ratings_df["rating"] > 0].drop_duplicates(subset=["user_id", "item_id"])

Cleans book catalogue

In [10]:
books_df = books_raw.drop_duplicates(subset=["item_id"]).copy()
books_df["item_id"] = books_df["item_id"].astype(str).str.strip()
for col in ["title", "author", "publisher", "year"]:
    if col not in books_df.columns:
        books_df[col] = ""
    books_df[col] = books_df[col].fillna("").astype(str).str.strip()

# Keep only interactions for books present in BX-Books
ratings_df = ratings_df[ratings_df["item_id"].isin(set(books_df["item_id"]))]    

Filter to users and books with at least 5 explicit ratings so evaluation & SVD are well-supported

In [11]:
active_users = ratings_df["user_id"].value_counts()
active_items = ratings_df["item_id"].value_counts()
ratings_df = ratings_df[
    ratings_df["user_id"].isin(active_users[active_users >= 5].index)
    & ratings_df["item_id"].isin(active_items[active_items >= 5].index)
].reset_index(drop=True)

books_df = books_df[books_df["item_id"].isin(set(ratings_df["item_id"]))].reset_index(drop=True)
books_df["metadata"] = (
    books_df["title"] + " " + books_df["author"] + " " + books_df["publisher"] + " " + books_df["year"]
).str.lower()

n_users = ratings_df["user_id"].nunique()
n_items = ratings_df["item_id"].nunique()
n_inter = len(ratings_df)
sparsity = 1.0 - (n_inter / max(1, n_users * n_items))

print(f"\nCleaned counts -> Users: {n_users} | Items: {n_items} | Interactions: {n_inter} | Sparsity: {sparsity:.4f}")
print("\nSample cleaned ratings:")
print(ratings_df.head(5))
print("\nSample cleaned books:")
print(books_df[["item_id", "title", "author", "year"]].head(5))


Cleaned counts -> Users: 12293 | Items: 13769 | Interactions: 145890 | Sparsity: 0.9991

Sample cleaned ratings:
  user_id     item_id  rating
0  276747  0060517794       9
1  276747  0671537458       9
2  276747  0679776818       8
3  276813  8426449476       8
4  276822  0060096195      10

Sample cleaned books:
      item_id                                              title  \
0  0002005018                                       Clara Callan   
1  0374157065  Flu: The Story of the Great Influenza Pandemic...   
2  0399135782                             The Kitchen God's Wife   
3  0440234743                                      The Testament   
4  0452264464               Beloved (Plume Contemporary Fiction)   

                 author  year  
0  Richard Bruce Wright  2001  
1      Gina Bari Kolata  1999  
2               Amy Tan  1991  
3          John Grisham  1999  
4         Toni Morrison  1994  


Book-Crossing typically starts with around 1.15 million total rows in BX-Book-Ratings (many of which are 0 implicit ratings), ~271k books, and ~278k users. Filtering to explicit ratings (rating > 0), matching BX-Books, and requiring >= 5 ratings per user and item will shrink those numbers—just make sure the final filtered counts stay comfortably above the assignment's >= 10,000 interactions, >= 500 users, and >= 500 items thresholds (if >= 5 filters too aggressively on my specific CSV, lowering the threshold from 5 to 3 keeps more rows).

Sparsity value: The number 0.9991 (99.91% empty cells), which is classic for book-rating matrices : most users haven't read most books. Measuring sparsity explains why simple overlap methods struggle and why latent factors or content features help.

#### TRAIN /TEST Split and popularity baseline

Split each active user's ratings (80% train / 20% test) and rank unseen books by popularity (pop_count, breaking ties with avg_rating)

In [12]:
def train_test_split_user(df, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_parts, test_parts = [], []
    for _, group in df.groupby("user_id"):
        if len(group) >= 3:
            idx = group.index.to_numpy().copy()
            rng.shuffle(idx)
            n_test = max(1, int(np.floor(len(idx) * test_ratio)))
            test_parts.append(df.loc[idx[:n_test]])
            train_parts.append(df.loc[idx[n_test:]])
        else:
            train_parts.append(group)
    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame(columns=df.columns)
    return train_df, test_df


train_df, test_df = train_test_split_user(ratings_df)

item_to_title = dict(zip(books_df["item_id"], books_df["title"]))
item_to_author = dict(zip(books_df["item_id"], books_df["author"]))
item_to_idx = {iid: i for i, iid in enumerate(books_df["item_id"])}
idx_to_item = {i: iid for iid, i in item_to_idx.items()}

pop_stats = (
    train_df.groupby("item_id")
    .agg(pop_count=("user_id", "count"), avg_rating=("rating", "mean"))
    .reset_index()
)
pop_ranked_df = (
    books_df[["item_id", "title", "author"]]
    .merge(pop_stats, on="item_id", how="left")
    .fillna({"pop_count": 0, "avg_rating": 0.0})
    .sort_values(["pop_count", "avg_rating"], ascending=False)
    .reset_index(drop=True)
)
pop_rank_lookup = {iid: rank + 1 for rank, iid in enumerate(pop_ranked_df["item_id"])}


def recommend_popularity(user_id, seen_items, top_k=5):
    recs = []
    for _, row in pop_ranked_df.iterrows():
        iid = row["item_id"]
        if iid not in seen_items:
            recs.append((
                iid,
                row["title"],
                float(row["pop_count"]),
                f"Popular overall ({int(row['pop_count'])} ratings, avg {row['avg_rating']:.1f}/10)"
            ))
        if len(recs) == top_k:
            break
    return recs


sample_uid = train_df["user_id"].iloc[0]
seen_sample = set(train_df[train_df["user_id"] == sample_uid]["item_id"])
print(f"\n--- Popularity Baseline for {sample_uid} ---")
for iid, title, score, reason in recommend_popularity(sample_uid, seen_sample, top_k=5):
    print(f"  * {title} [{iid}] — {reason}")


--- Popularity Baseline for 100004 ---
  * The Lovely Bones: A Novel [0316666343] — Popular overall (360 ratings, avg 8.2/10)
  * The Da Vinci Code [0385504209] — Popular overall (284 ratings, avg 8.5/10)
  * Wild Animus [0971880107] — Popular overall (208 ratings, avg 3.9/10)
  * The Red Tent (Bestselling Backlist) [0312195516] — Popular overall (194 ratings, avg 8.4/10)
  * Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)) [059035342X] — Popular overall (186 ratings, avg 9.0/10)


Because pop_count is listed first, the baseline ranks books by how many people rated them, not by how high their average rating is. Wild Animus (Rich Shapero) is infamous in the Book-Crossing dataset for having a massive number of ratings alongside a notoriously low average rating (~3.9/10), so ranking purely by interaction count pushes it into the Top 5 even though readers generally disliked it.

This is a classic insight to highlight in your EDA and baseline discussion: high interaction volume (pop_count) does not equal high quality (avg_rating). If you want a quality-aware popularity baseline, you could rank by avg_rating (with a minimum rating threshold so a 1-person 10/10 doesn't win) or a Bayesian/weighted rating score.

These are the top rated books that the test hasnt seen in training ,showing total rating count and mean score .

We are seeing high frequency tiles(8-9) out of 10 because popularity ignores individual reading taste ,it serves as our baseline bench 

#### Content bade filtering (TF-IDF + Cosine Similarity)

Built TF-IDF vectors from book metadata (title, author, publisher, year), weights books rated >= 7.0 to form a user taste vector, and ranks unseen books by cosine similarity.

In [13]:
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(books_df["metadata"])


def recommend_content_based(user_id, user_history_df, top_k=5, like_threshold=7.0):
    seen_items = set(user_history_df["item_id"])
    liked = user_history_df[user_history_df["rating"] >= like_threshold]
    if liked.empty:
        liked = user_history_df
    if liked.empty:
        return recommend_popularity(user_id, seen_items, top_k=top_k)

    profile = np.zeros((1, tfidf_matrix.shape[1]), dtype=float)
    valid_liked_titles = []
    for _, row in liked.iterrows():
        iid = row["item_id"]
        if iid in item_to_idx:
            weight = max(float(row["rating"]), 1.0)
            profile += weight * tfidf_matrix[item_to_idx[iid]].toarray()
            valid_liked_titles.append(item_to_title.get(iid, iid))

    if np.allclose(profile, 0.0):
        return recommend_popularity(user_id, seen_items, top_k=top_k)

    sims = cosine_similarity(profile, tfidf_matrix).flatten()
    ranked_indices = np.argsort(-sims)
    anchor_title = valid_liked_titles[0] if valid_liked_titles else "your reading history"

    recs = []
    for idx in ranked_indices:
        iid = idx_to_item[idx]
        if iid not in seen_items:
            recs.append((
                iid,
                item_to_title.get(iid, iid),
                float(sims[idx]),
                f"Similar metadata profile to '{anchor_title}' (sim={sims[idx]:.2f})"
            ))
        if len(recs) == top_k:
            break
    return recs


print(f"\n--- Content-Based for {sample_uid} ---")
hist_sample = train_df[train_df["user_id"] == sample_uid]
for iid, title, score, reason in recommend_content_based(sample_uid, hist_sample, top_k=5):
    print(f"  * {title} [{iid}] — {reason}")


--- Content-Based for 100004 ---
  * The Hunt for Red October [0425120279] — Similar metadata profile to 'The Hunt for Red October' (sim=0.51)
  * Harry Potter and the Chamber of Secrets (Book 2) [0439420105] — Similar metadata profile to 'The Hunt for Red October' (sim=0.49)
  * Harry Potter and the Chamber of Secrets (Book 2) [0439064864] — Similar metadata profile to 'The Hunt for Red October' (sim=0.49)
  * Harry Potter and the Chamber of Secrets Postcard Book [0439425220] — Similar metadata profile to 'The Hunt for Red October' (sim=0.45)
  * The Hunt for Red October (Jack Ryan Novels) [0425133516] — Similar metadata profile to 'The Hunt for Red October' (sim=0.44)


The five unseen books ranked by cosine similarity (0.00 to 1.00) to the user's liked books.

Cosine similarities on short metadata strings are often small (~0.05–0.40) unless the user liked a book by a prolific author or publisher with repeating terms in books_df. 

discussing content limitations vs. rich plot/genre features.

Explain why your TF-IDF content-based model scored or behaved the way it did—relying on short metadata fields (title, author, publisher, year) gives sparse lexical overlap compared to rich plot summaries, blurbs, or standardized genre tags.

#### Matrix Factorization 

Centers each user's ratings around their mean score, extracts n_components latent factors via TruncatedSVD, and predicts scores for unseen books.

In [15]:
pivot = train_df.pivot_table(index="user_id", columns="item_id", values="rating", fill_value=0.0)
user_means = train_df.groupby("user_id")["rating"].mean().reindex(pivot.index).fillna(train_df["rating"].mean())
centered_matrix = pivot.sub(user_means, axis=0).where(pivot > 0, 0.0)

n_components = min(20, min(centered_matrix.shape) - 1) if min(centered_matrix.shape) > 1 else 1
svd = TruncatedSVD(n_components=n_components, random_state=42)
U = svd.fit_transform(centered_matrix)
Vt = svd.components_
pred_matrix = pd.DataFrame(
    np.dot(U, Vt) + user_means.to_numpy()[:, None],
    index=pivot.index,
    columns=pivot.columns,
)


def recommend_svd(user_id, user_history_df, top_k=5):
    seen_items = set(user_history_df["item_id"])
    if user_id not in pred_matrix.index:
        return recommend_popularity(user_id, seen_items, top_k=top_k)

    scores = pred_matrix.loc[user_id].sort_values(ascending=False)
    recs = []
    for iid, score in scores.items():
        if iid not in seen_items and iid in item_to_title:
            recs.append((
                iid,
                item_to_title[iid],
                float(score),
                f"Latent-factor SVD predicted score: {score:.2f}/10"
            ))
        if len(recs) == top_k:
            break
    return recs


print(f"\n--- SVD Recommendations for {sample_uid} ---")
for iid, title, score, reason in recommend_svd(sample_uid, hist_sample, top_k=5):
    print(f"  * {title} [{iid}] — {reason}")


--- SVD Recommendations for 100004 ---
  * Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)) [059035342X] — Latent-factor SVD predicted score: 9.53/10
  * Harry Potter and the Order of the Phoenix (Book 5) [043935806X] — Latent-factor SVD predicted score: 9.51/10
  * The Stand: Complete and Uncut [0451169530] — Latent-factor SVD predicted score: 9.51/10
  * A Time to Kill [0440211727] — Latent-factor SVD predicted score: 9.51/10
  * Harry Potter and the Goblet of Fire (Book 4) [0439139597] — Latent-factor SVD predicted score: 9.51/10


Five unseen books ranked by predicted score (pred_score), along with the latent-factor explanation.

Predicted scores generally cluster around the user's mean rating plus latent taste adjustments (roughly 6–10 for top picks), capturing collaborative reader patterns that don't depend on matching words in book titles.

#### Evaluation Metrics and sample user outputs 

Evaluates Popularity, Content-Based, and SVD on held-out test ratings (rating >= 7.0) using Precision@5, Recall@5, Catalog_Coverage, and Popularity_Bias, then prints Top-5 SVD picks for three users.

In [16]:
def evaluate_recommender(model_name, k=5, rel_threshold=7.0):
    user_train_map = {uid: grp for uid, grp in train_df.groupby("user_id")}
    user_test_rel = {
        uid: set(grp[grp["rating"] >= rel_threshold]["item_id"])
        for uid, grp in test_df.groupby("user_id")
    }

    eval_users = [u for u, rel in user_test_rel.items() if len(rel) > 0 and u in user_train_map]
    if not eval_users:
        eval_users = list(user_train_map.keys())[:50]

    precisions, recalls, all_recs, avg_pop_ranks = [], [], [], []
    total_catalog = len(books_df)
    total_ranks = max(1, len(pop_ranked_df))

    for uid in eval_users:
        hist = user_train_map[uid]
        seen = set(hist["item_id"])
        if model_name == "Popularity":
            recs = recommend_popularity(uid, seen, top_k=k)
        elif model_name == "Content-Based":
            recs = recommend_content_based(uid, hist, top_k=k)
        else:
            recs = recommend_svd(uid, hist, top_k=k)

        rec_ids = [r[0] for r in recs]
        all_recs.append(rec_ids)

        if rec_ids:
            pop_scores = [1.0 - ((pop_rank_lookup.get(iid, total_ranks) - 1) / total_ranks) for iid in rec_ids]
            avg_pop_ranks.append(float(np.mean(pop_scores)))

        rel_set = user_test_rel.get(uid, set())
        if rel_set and rec_ids:
            hits = len(set(rec_ids) & rel_set)
            precisions.append(hits / k)
            recalls.append(hits / len(rel_set))

    unique_rec_items = {iid for rlist in all_recs for iid in rlist}
    coverage = len(unique_rec_items) / max(1, total_catalog)
    pop_bias = float(np.mean(avg_pop_ranks)) if avg_pop_ranks else 0.0

    return {
        "Model": model_name,
        f"Precision@{k}": round(float(np.mean(precisions)), 4) if precisions else 0.0,
        f"Recall@{k}": round(float(np.mean(recalls)), 4) if recalls else 0.0,
        "Catalog_Coverage": round(coverage, 4),
        "Popularity_Bias": round(pop_bias, 4),
    }


results_df = pd.DataFrame([
    evaluate_recommender("Popularity", k=5),
    evaluate_recommender("Content-Based", k=5),
    evaluate_recommender("SVD", k=5),
])

print("\n=== MODEL COMPARISON (Top-5) ===")
print(results_df.to_string(index=False))

for uid in train_df["user_id"].unique()[:3]:
    hist = train_df[train_df["user_id"] == uid]
    print(f"\n--- Example User: {uid} (rated {len(hist)} books in train) ---")
    for iid, title, score, reason in recommend_svd(uid, hist, top_k=5):
        print(f"  * {title} [{iid}] — {reason}")


=== MODEL COMPARISON (Top-5) ===
        Model  Precision@5  Recall@5  Catalog_Coverage  Popularity_Bias
   Popularity       0.0048    0.0121            0.0007           0.9998
Content-Based       0.0180    0.0458            0.6310           0.6504
          SVD       0.0038    0.0080            0.0304           0.9068

--- Example User: 100004 (rated 4 books in train) ---
  * Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)) [059035342X] — Latent-factor SVD predicted score: 9.53/10
  * Harry Potter and the Order of the Phoenix (Book 5) [043935806X] — Latent-factor SVD predicted score: 9.51/10
  * The Stand: Complete and Uncut [0451169530] — Latent-factor SVD predicted score: 9.51/10
  * A Time to Kill [0440211727] — Latent-factor SVD predicted score: 9.51/10
  * Harry Potter and the Goblet of Fire (Book 4) [0439139597] — Latent-factor SVD predicted score: 9.51/10

--- Example User: 100009 (rated 12 books in train) ---
  * The Secret Life of Bees [0142001740] — Latent-f

A 3-row comparison table (Popularity, Content-Based, SVD) across Precision@5, Recall@5, Catalog_Coverage, and Popularity_Bias, followed by Top-5 SVD recommendations for 3 example users.

Precision@5 and Recall@5 are often small decimals (e.g., 0.00 to 0.08) on large sparse book catalogs because there are many thousands of possible books and only a small set of held-out test books per user—focus on comparing which model ranks relevant books higher.

Popularity usually has the highest Popularity_Bias (closer to 1.0) and lowest Catalog_Coverage, whereas personalized models typically reach more diverse catalog items.